In [4]:
import time
import pickle

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from tensorflow.keras import Input
from tensorflow.keras.models import Sequential

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from tensorflow.keras.layers import (
    Embedding,
    Dense,
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    GlobalAveragePooling1D,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.preprocessing.sequence import pad_sequences

In [5]:
data = pd.read_csv("imdb_cleaned.csv")
print(data.shape)

(49582, 4)


In [6]:
X = data["clean_review"]
y = data["label"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_test,
    y_test,
    test_size=0.50,
    random_state=42,
    stratify=y_test
)

print(len(X_train))
print(len(X_val))
print(len(X_test))

39665
4958
4959


In [8]:
import pickle

with open("tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)

print(len(tokenizer.word_index))

90662


In [9]:
MAX_SEQUENCE_LENGTH = 500

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_val_sequences = tokenizer.texts_to_sequences(X_val)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

X_train_integer = pad_sequences(
    X_train_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_val_integer = pad_sequences(
    X_val_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_test_integer = pad_sequences(
    X_test_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

y_train = np.asarray(y_train)
y_val = np.asarray(y_val)
y_test = np.asarray(y_test)

print(X_train_integer.shape)
print(X_val_integer.shape)
print(X_test_integer.shape)

(39665, 500)
(4958, 500)
(4959, 500)


In [10]:
def evaluate_model(model, X_test, y_test, model_name, batch_size=64):

    # Generate probabilities
    probabilities = model.predict(
        X_test,
        batch_size=batch_size,
        verbose=0
    ).ravel()

    # Convert probabilities to class predictions
    predictions = (probabilities >= 0.5).astype(int)

    # metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test,  predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
    roc_auc = roc_auc_score(y_test, probabilities)

    # Confusion Matrix
    cm = confusion_matrix(y_test, predictions)

    # Classification Report
    report = classification_report(
        y_test,
        predictions,
        target_names=["Negative", "Positive"],
        digits=4
    )

    # Print results
    print("=" * 60)
    print(f"{model_name} RESULTS")
    print("=" * 60)

    print(f"Accuracy : {accuracy:.5f}")
    print(f"Precision: {precision:.5f}")
    print(f"Recall   : {recall:.5f}")
    print(f"F1 Score : {f1:.5f}")
    print(f"ROC-AUC  : {roc_auc:.5f}")

    print("\nClassification Report")
    print("-" * 60)
    print(report)

    print("Confusion Matrix")
    print(cm)

    # Return everything for later comparison
    results = {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    }

    return results, probabilities, predictions, cm

In [11]:
ann_results_list = []
ann_results_list.append({
    "Model": "ANN Baseline",
    "Accuracy": 0.75015,
    "Precision": 0.96712,
    "Recall": 0.51989,
    "F1 Score": 0.67625,
    "ROC-AUC": 0.95225
})

In [12]:
ann_df = pd.DataFrame(ann_results_list)
display(ann_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.75015,0.96712,0.51989,0.67625,0.95225


**ANN EarlyStopping**

In [13]:
VOCAB_SIZE = 30000
EMBEDDING_DIM = 128
MAX_SEQUENCE_LENGTH = 500

In [14]:
ann_early = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    GlobalAveragePooling1D(),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_early.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_early.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,864,833 (14.74 MB)

 Trainable params: 3,864,833 (14.74 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping_ann = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [17]:
history_ann_early = ann_early.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping_ann],
    verbose=1
)

Epoch 1/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.7365 - loss: 0.5039 - val_accuracy: 0.8657 - val_loss: 0.3233
Epoch 2/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8562 - loss: 0.3312 - val_accuracy: 0.8842 - val_loss: 0.2820
Epoch 3/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8810 - loss: 0.2798 - val_accuracy: 0.7862 - val_loss: 0.4881
Epoch 4/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8969 - loss: 0.2512 - val_accuracy: 0.8511 - val_loss: 0.3346
Epoch 5/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9077 - loss: 0.2317 - val_accuracy: 0.8633 - val_loss: 0.3081
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.


In [18]:
ann_early_results, ann_early_probabilities, ann_early_predictions, ann_early_cm = evaluate_model(
    ann_early,
    X_test_integer,
    y_test,
    "ANN EarlyStopping",
    batch_size=64
)

ANN EarlyStopping RESULTS
Accuracy : 0.89433
Precision: 0.91403
Recall   : 0.87143
F1 Score : 0.89223
ROC-AUC  : 0.95522

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8763    0.9174    0.8964      2470
    Positive     0.9140    0.8714    0.8922      2489

    accuracy                         0.8943      4959
   macro avg     0.8951    0.8944    0.8943      4959
weighted avg     0.8952    0.8943    0.8943      4959

Confusion Matrix
[[2266  204]
 [ 320 2169]]


In [19]:
ann_results_list.append(ann_early_results)

ann_df = pd.DataFrame(ann_results_list)
display(ann_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.750150,0.967120,0.519890,0.676250,0.952250
1,ANN EarlyStopping,0.894334,0.914033,0.871434,0.892225,0.955224


**ANN Dropout**

In [20]:
ann_dropout = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    GlobalAveragePooling1D(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

ann_dropout.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_dropout.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,864,833 (14.74 MB)

 Trainable params: 3,864,833 (14.74 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
history_ann_dropout = ann_dropout.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.6559 - loss: 0.5904 - val_accuracy: 0.8269 - val_loss: 0.3925
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8380 - loss: 0.3656 - val_accuracy: 0.7497 - val_loss: 0.5098
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8756 - loss: 0.3018 - val_accuracy: 0.8584 - val_loss: 0.3284
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8923 - loss: 0.2643 - val_accuracy: 0.8122 - val_loss: 0.4045
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9072 - loss: 0.2337 - val_accuracy: 0.8977 - val_loss: 0.2590
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9102 - loss: 0.2231 - val_accuracy: 0.8078 - val_loss: 0.4941
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9210 - loss: 0.2043 - val_accuracy: 0.8328 - val_loss: 0.3839
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9258 - loss: 0.1918 - val_accuracy: 0

In [22]:
ann_dropout_results, ann_dropout_probabilities, ann_dropout_predictions, ann_dropout_cm = evaluate_model(
    ann_dropout,
    X_test_integer,
    y_test,
    "ANN Dropout",
    batch_size=64
)

ANN Dropout RESULTS
Accuracy : 0.85763
Precision: 0.94597
Recall   : 0.75974
F1 Score : 0.84269
ROC-AUC  : 0.95714

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.7980    0.9563    0.8700      2470
    Positive     0.9460    0.7597    0.8427      2489

    accuracy                         0.8576      4959
   macro avg     0.8720    0.8580    0.8563      4959
weighted avg     0.8723    0.8576    0.8563      4959

Confusion Matrix
[[2362  108]
 [ 598 1891]]


In [23]:
ann_results_list.append(ann_dropout_results)

ann_df = pd.DataFrame(ann_results_list)
display(ann_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.750150,0.967120,0.519890,0.676250,0.952250
1,ANN EarlyStopping,0.894334,0.914033,0.871434,0.892225,0.955224
2,ANN Dropout,0.857633,0.945973,0.759743,0.842692,0.957139


**ANN Batch Normalization**

In [24]:
ann_bn = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    GlobalAveragePooling1D(),
    Dense(128, activation="relu"),
    BatchNormalization(),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(1, activation="sigmoid")
])

ann_bn.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_bn.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,865,601 (14.75 MB)

 Trainable params: 3,865,217 (14.74 MB)

 Non-trainable params: 384 (1.50 KB)

In [25]:
history_ann_bn = ann_bn.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.8576 - loss: 0.3252 - val_accuracy: 0.8808 - val_loss: 0.2952
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9296 - loss: 0.1846 - val_accuracy: 0.6251 - val_loss: 2.2761
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9511 - loss: 0.1275 - val_accuracy: 0.7652 - val_loss: 0.8952
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9660 - loss: 0.0936 - val_accuracy: 0.8217 - val_loss: 0.5311
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9741 - loss: 0.0725 - val_accuracy: 0.8598 - val_loss: 0.4883
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9793 - loss: 0.0577 - val_accuracy: 0.8669 - val_loss: 0.5114
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9829 - loss: 0.0488 - val_accuracy: 0.8594 - val_loss: 0.5640
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9840 - loss: 0.0434 - val_accuracy: 0.

In [26]:
ann_bn_results, ann_bn_probabilities, ann_bn_predictions, ann_bn_cm = evaluate_model(
    ann_bn,
    X_test_integer,
    y_test,
    "ANN Batch Normalization",
    batch_size=64
)

ANN Batch Normalization RESULTS
Accuracy : 0.71426
Precision: 0.91422
Recall   : 0.47529
F1 Score : 0.62543
ROC-AUC  : 0.89316

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.6437    0.9551    0.7690      2470
    Positive     0.9142    0.4753    0.6254      2489

    accuracy                         0.7143      4959
   macro avg     0.7789    0.7152    0.6972      4959
weighted avg     0.7795    0.7143    0.6970      4959

Confusion Matrix
[[2359  111]
 [1306 1183]]


In [28]:
ann_results_list.append(ann_bn_results)

ann_df = pd.DataFrame(ann_results_list)
display(ann_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.750150,0.967120,0.519890,0.676250,0.952250
1,ANN EarlyStopping,0.894334,0.914033,0.871434,0.892225,0.955224
2,ANN Dropout,0.857633,0.945973,0.759743,0.842692,0.957139
3,ANN Batch Normalization,0.714257,0.914219,0.475291,0.625430,0.893157


**ANN Learning Rate**

In [29]:
ann_lr = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    GlobalAveragePooling1D(),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_lr.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_lr.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_3      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,864,833 (14.74 MB)

 Trainable params: 3,864,833 (14.74 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
history_ann_lr = ann_lr.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.6793 - loss: 0.5623 - val_accuracy: 0.7802 - val_loss: 0.4351
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8613 - loss: 0.3283 - val_accuracy: 0.8445 - val_loss: 0.3432
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8844 - loss: 0.2801 - val_accuracy: 0.8564 - val_loss: 0.3261
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9019 - loss: 0.2414 - val_accuracy: 0.8626 - val_loss: 0.3147
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9101 - loss: 0.2227 - val_accuracy: 0.8897 - val_loss: 0.2693
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9200 - loss: 0.2028 - val_accuracy: 0.8973 - val_loss: 0.2558
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9308 - loss: 0.1789 - val_accuracy: 0.8707 - val_loss: 0.3080
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9317 - loss: 0.1750 - val_accuracy: 0.

In [31]:
ann_lr_results, ann_lr_probabilities, ann_lr_predictions, ann_lr_cm = evaluate_model(
    ann_lr,
    X_test_integer,
    y_test,
    "ANN Learning Rate 0.0005",
    batch_size=64
)

ANN Learning Rate 0.0005 RESULTS
Accuracy : 0.85804
Precision: 0.93600
Recall   : 0.76979
F1 Score : 0.84480
ROC-AUC  : 0.95605

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8032    0.9470    0.8692      2470
    Positive     0.9360    0.7698    0.8448      2489

    accuracy                         0.8580      4959
   macro avg     0.8696    0.8584    0.8570      4959
weighted avg     0.8699    0.8580    0.8569      4959

Confusion Matrix
[[2339  131]
 [ 573 1916]]


In [32]:
ann_results_list.append(ann_lr_results)

ann_df = pd.DataFrame(ann_results_list)
display(ann_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.750150,0.967120,0.519890,0.676250,0.952250
1,ANN EarlyStopping,0.894334,0.914033,0.871434,0.892225,0.955224
2,ANN Dropout,0.857633,0.945973,0.759743,0.842692,0.957139
3,ANN Batch Normalization,0.714257,0.914219,0.475291,0.625430,0.893157
4,ANN Learning Rate 0.0005,0.858036,0.936004,0.769787,0.844797,0.956050


**ANN ReduceLR**

In [33]:
ann_reducelr = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    GlobalAveragePooling1D(),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_reducelr.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_reducelr.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_4      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,864,833 (14.74 MB)

 Trainable params: 3,864,833 (14.74 MB)

 Non-trainable params: 0 (0.00 B)

In [34]:
reduce_lr_ann = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [35]:
history_ann_reducelr = ann_reducelr.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    callbacks=[reduce_lr_ann],
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.7096 - loss: 0.5284 - val_accuracy: 0.7763 - val_loss: 0.4455 - learning_rate: 0.0010
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8520 - loss: 0.3342 - val_accuracy: 0.8616 - val_loss: 0.3181 - learning_rate: 0.0010
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8763 - loss: 0.2962 - val_accuracy: 0.8643 - val_loss: 0.3079 - learning_rate: 0.0010
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8963 - loss: 0.2521 - val_accuracy: 0.8618 - val_loss: 0.3177 - learning_rate: 0.0010
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9140 - loss: 0.2149 - val_accuracy: 0.8957 - val_loss: 0.2558 - learning_rate: 0.0010
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9203 - loss: 0.1990 - val_accuracy: 0.8927 - val_loss: 0.2665 - learning_rate: 0.0010
Epoch 7/10
610/620 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9276 - loss: 0.1844
Ep

In [36]:
ann_reducelr_results, ann_reducelr_probabilities, ann_reducelr_predictions, ann_reducelr_cm = evaluate_model(
    ann_reducelr,
    X_test_integer,
    y_test,
    "ANN ReduceLR",
    batch_size=64
)

ANN ReduceLR RESULTS
Accuracy : 0.89292
Precision: 0.92089
Recall   : 0.86059
F1 Score : 0.88972
ROC-AUC  : 0.95953

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8682    0.9255    0.8959      2470
    Positive     0.9209    0.8606    0.8897      2489

    accuracy                         0.8929      4959
   macro avg     0.8946    0.8930    0.8928      4959
weighted avg     0.8947    0.8929    0.8928      4959

Confusion Matrix
[[2286  184]
 [ 347 2142]]


In [37]:
ann_results_list.append(ann_reducelr_results)

ann_df = pd.DataFrame(ann_results_list)
display(ann_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.750150,0.967120,0.519890,0.676250,0.952250
1,ANN EarlyStopping,0.894334,0.914033,0.871434,0.892225,0.955224
2,ANN Dropout,0.857633,0.945973,0.759743,0.842692,0.957139
3,ANN Batch Normalization,0.714257,0.914219,0.475291,0.625430,0.893157
4,ANN Learning Rate 0.0005,0.858036,0.936004,0.769787,0.844797,0.956050
5,ANN ReduceLR,0.892922,0.920894,0.860587,0.889720,0.959529


**ANN Batch Size 32**

In [38]:
ann_batch32 = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    GlobalAveragePooling1D(),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_batch32.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_batch32.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_5      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,864,833 (14.74 MB)

 Trainable params: 3,864,833 (14.74 MB)

 Non-trainable params: 0 (0.00 B)

In [39]:
history_ann_batch32 = ann_batch32.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

Epoch 1/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.7302 - loss: 0.4998 - val_accuracy: 0.8735 - val_loss: 0.3126
Epoch 2/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.8665 - loss: 0.3129 - val_accuracy: 0.8887 - val_loss: 0.2727
Epoch 3/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.8902 - loss: 0.2652 - val_accuracy: 0.7961 - val_loss: 0.4572
Epoch 4/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9078 - loss: 0.2256 - val_accuracy: 0.8423 - val_loss: 0.3555
Epoch 5/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9170 - loss: 0.2083 - val_accuracy: 0.8895 - val_loss: 0.2731
Epoch 6/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9171 - loss: 0.2078 - val_accuracy: 0.8987 - val_loss: 0.2686
Epoch 7/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9295 - loss: 0.1796 - val_accuracy: 0.8822 - val_loss: 0.3150
Epoch 8/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9324 - loss: 0.1725 - 

In [40]:
ann_batch32_results, ann_batch32_probabilities, ann_batch32_predictions, ann_batch32_cm = evaluate_model(
    ann_batch32,
    X_test_integer,
    y_test,
    "ANN Batch Size 32",
    batch_size=32
)

ANN Batch Size 32 RESULTS
Accuracy : 0.89575
Precision: 0.91290
Recall   : 0.87585
F1 Score : 0.89399
ROC-AUC  : 0.95388

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8798    0.9158    0.8974      2470
    Positive     0.9129    0.8759    0.8940      2489

    accuracy                         0.8957      4959
   macro avg     0.8964    0.8958    0.8957      4959
weighted avg     0.8964    0.8957    0.8957      4959

Confusion Matrix
[[2262  208]
 [ 309 2180]]


In [41]:
ann_results_list.append(ann_batch32_results)

ann_df = pd.DataFrame(ann_results_list)
display(ann_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.750150,0.967120,0.519890,0.676250,0.952250
1,ANN EarlyStopping,0.894334,0.914033,0.871434,0.892225,0.955224
2,ANN Dropout,0.857633,0.945973,0.759743,0.842692,0.957139
3,ANN Batch Normalization,0.714257,0.914219,0.475291,0.625430,0.893157
4,ANN Learning Rate 0.0005,0.858036,0.936004,0.769787,0.844797,0.956050
5,ANN ReduceLR,0.892922,0.920894,0.860587,0.889720,0.959529
6,ANN Batch Size 32,0.895745,0.912898,0.875854,0.893992,0.953881


**ANN Batch Size 128**

In [42]:
ann_batch128 = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    GlobalAveragePooling1D(),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_batch128.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_batch128.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_6      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,864,833 (14.74 MB)

 Trainable params: 3,864,833 (14.74 MB)

 Non-trainable params: 0 (0.00 B)

In [43]:
history_ann_batch128 = ann_batch128.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=128,
    verbose=1
)

Epoch 1/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.6310 - loss: 0.6167 - val_accuracy: 0.6402 - val_loss: 0.6399
Epoch 2/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8382 - loss: 0.3677 - val_accuracy: 0.8300 - val_loss: 0.3627
Epoch 3/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8773 - loss: 0.2918 - val_accuracy: 0.8241 - val_loss: 0.4013
Epoch 4/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8904 - loss: 0.2668 - val_accuracy: 0.8927 - val_loss: 0.2620
Epoch 5/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9021 - loss: 0.2391 - val_accuracy: 0.8969 - val_loss: 0.2560
Epoch 6/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9080 - loss: 0.2266 - val_accuracy: 0.8818 - val_loss: 0.2867
Epoch 7/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9130 - loss: 0.2114 - val_accuracy: 0.8921 - val_loss: 0.2672
Epoch 8/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9210 - loss: 0.1976 - val_accuracy: 0

In [44]:
ann_batch128_results, ann_batch128_probabilities, ann_batch128_predictions, ann_batch128_cm = evaluate_model(
    ann_batch128,
    X_test_integer,
    y_test,
    "ANN Batch Size 128",
    batch_size=128
)

ANN Batch Size 128 RESULTS
Accuracy : 0.89736
Precision: 0.90707
Recall   : 0.88630
F1 Score : 0.89657
ROC-AUC  : 0.95759

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8880    0.9085    0.8981      2470
    Positive     0.9071    0.8863    0.8966      2489

    accuracy                         0.8974      4959
   macro avg     0.8975    0.8974    0.8974      4959
weighted avg     0.8976    0.8974    0.8973      4959

Confusion Matrix
[[2244  226]
 [ 283 2206]]


In [45]:
ann_results_list.append(ann_batch128_results)

ann_df = pd.DataFrame(ann_results_list)
display(ann_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.750150,0.967120,0.519890,0.676250,0.952250
1,ANN EarlyStopping,0.894334,0.914033,0.871434,0.892225,0.955224
2,ANN Dropout,0.857633,0.945973,0.759743,0.842692,0.957139
3,ANN Batch Normalization,0.714257,0.914219,0.475291,0.625430,0.893157
4,ANN Learning Rate 0.0005,0.858036,0.936004,0.769787,0.844797,0.956050
5,ANN ReduceLR,0.892922,0.920894,0.860587,0.889720,0.959529
6,ANN Batch Size 32,0.895745,0.912898,0.875854,0.893992,0.953881
7,ANN Batch Size 128,0.897358,0.907072,0.886300,0.896566,0.957588


**ANN SGD**

In [46]:
ann_sgd = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    GlobalAveragePooling1D(),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_sgd.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_sgd.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_7      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,864,833 (14.74 MB)

 Trainable params: 3,864,833 (14.74 MB)

 Non-trainable params: 0 (0.00 B)

In [47]:
history_ann_sgd = ann_sgd.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 26s 41ms/step - accuracy: 0.5029 - loss: 0.6932 - val_accuracy: 0.4990 - val_loss: 0.6932
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 24s 38ms/step - accuracy: 0.5010 - loss: 0.6932 - val_accuracy: 0.4923 - val_loss: 0.6932
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 23s 37ms/step - accuracy: 0.5019 - loss: 0.6932 - val_accuracy: 0.4911 - val_loss: 0.6932
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 23s 38ms/step - accuracy: 0.4973 - loss: 0.6931 - val_accuracy: 0.4899 - val_loss: 0.6932
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 24s 38ms/step - accuracy: 0.4967 - loss: 0.6931 - val_accuracy: 0.4893 - val_loss: 0.6932
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 23s 37ms/step - accuracy: 0.4903 - loss: 0.6931 - val_accuracy: 0.4911 - val_loss: 0.6932
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 23s 37ms/step - accuracy: 0.4952 - loss: 0.6931 - val_accuracy: 0.4806 - val_loss: 0.6932
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 23s 37ms/step - accuracy: 0.4948 - loss: 0.6931 - 

In [48]:
ann_sgd_results, ann_sgd_probabilities, ann_sgd_predictions, ann_sgd_cm = evaluate_model(
    ann_sgd,
    X_test_integer,
    y_test,
    "ANN SGD",
    batch_size=64
)

ANN SGD RESULTS
Accuracy : 0.49647
Precision: 0.49865
Recall   : 0.59381
F1 Score : 0.54209
ROC-AUC  : 0.50285

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.4932    0.3984    0.4408      2470
    Positive     0.4987    0.5938    0.5421      2489

    accuracy                         0.4965      4959
   macro avg     0.4959    0.4961    0.4914      4959
weighted avg     0.4960    0.4965    0.4916      4959

Confusion Matrix
[[ 984 1486]
 [1011 1478]]


In [49]:
ann_results_list.append(ann_sgd_results)

ann_df = pd.DataFrame(ann_results_list)
display(ann_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.750150,0.967120,0.519890,0.676250,0.952250
1,ANN EarlyStopping,0.894334,0.914033,0.871434,0.892225,0.955224
2,ANN Dropout,0.857633,0.945973,0.759743,0.842692,0.957139
3,ANN Batch Normalization,0.714257,0.914219,0.475291,0.625430,0.893157
4,ANN Learning Rate 0.0005,0.858036,0.936004,0.769787,0.844797,0.956050
5,ANN ReduceLR,0.892922,0.920894,0.860587,0.889720,0.959529
6,ANN Batch Size 32,0.895745,0.912898,0.875854,0.893992,0.953881
7,ANN Batch Size 128,0.897358,0.907072,0.886300,0.896566,0.957588
8,ANN SGD,0.496471,0.498650,0.593813,0.542087,0.502848


**ANN RMSprop**

In [50]:
ann_rmsprop = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    GlobalAveragePooling1D(),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_rmsprop.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_rmsprop.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_8      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,864,833 (14.74 MB)

 Trainable params: 3,864,833 (14.74 MB)

 Non-trainable params: 0 (0.00 B)

In [51]:
history_ann_rmsprop = ann_rmsprop.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 84s 128ms/step - accuracy: 0.5032 - loss: 0.6929 - val_accuracy: 0.5095 - val_loss: 0.6922
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 76s 123ms/step - accuracy: 0.5309 - loss: 0.6876 - val_accuracy: 0.5313 - val_loss: 0.6750
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 76s 122ms/step - accuracy: 0.5763 - loss: 0.6691 - val_accuracy: 0.6864 - val_loss: 0.6332
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 76s 123ms/step - accuracy: 0.6212 - loss: 0.6449 - val_accuracy: 0.6660 - val_loss: 0.6107
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 76s 122ms/step - accuracy: 0.6557 - loss: 0.6192 - val_accuracy: 0.5494 - val_loss: 0.7628
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 76s 122ms/step - accuracy: 0.6873 - loss: 0.5910 - val_accuracy: 0.5470 - val_loss: 0.8129
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 75s 122ms/step - accuracy: 0.7092 - loss: 0.5585 - val_accuracy: 0.5726 - val_loss: 0.6814
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 76s 122ms/step - accuracy: 0.7254 - loss: 0

In [52]:
ann_rmsprop_results, ann_rmsprop_probabilities, ann_rmsprop_predictions, ann_rmsprop_cm = evaluate_model(
    ann_rmsprop,
    X_test_integer,
    y_test,
    "ANN RMSprop",
    batch_size=64
)

ANN RMSprop RESULTS
Accuracy : 0.83122
Precision: 0.78230
Recall   : 0.91965
F1 Score : 0.84543
ROC-AUC  : 0.91851

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.9016    0.7421    0.8141      2470
    Positive     0.7823    0.9196    0.8454      2489

    accuracy                         0.8312      4959
   macro avg     0.8420    0.8309    0.8298      4959
weighted avg     0.8417    0.8312    0.8298      4959

Confusion Matrix
[[1833  637]
 [ 200 2289]]


In [53]:
ann_results_list.append(ann_rmsprop_results)

ann_df = pd.DataFrame(ann_results_list)
display(ann_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.750150,0.967120,0.519890,0.676250,0.952250
1,ANN EarlyStopping,0.894334,0.914033,0.871434,0.892225,0.955224
2,ANN Dropout,0.857633,0.945973,0.759743,0.842692,0.957139
3,ANN Batch Normalization,0.714257,0.914219,0.475291,0.625430,0.893157
4,ANN Learning Rate 0.0005,0.858036,0.936004,0.769787,0.844797,0.956050
5,ANN ReduceLR,0.892922,0.920894,0.860587,0.889720,0.959529
6,ANN Batch Size 32,0.895745,0.912898,0.875854,0.893992,0.953881
7,ANN Batch Size 128,0.897358,0.907072,0.886300,0.896566,0.957588
8,ANN SGD,0.496471,0.498650,0.593813,0.542087,0.502848
9,ANN RMSprop,0.831216,0.782297,0.919646,0.845429,0.918507


**ANN Dim 256**

In [54]:
ann_dim256 = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    GlobalAveragePooling1D(),
    Dense(256, activation="relu"),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_dim256.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_dim256.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_9      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,889,537 (14.84 MB)

 Trainable params: 3,889,537 (14.84 MB)

 Non-trainable params: 0 (0.00 B)

In [55]:
history_ann_dim256 = ann_dim256.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.7233 - loss: 0.5114 - val_accuracy: 0.8384 - val_loss: 0.3585
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8538 - loss: 0.3364 - val_accuracy: 0.8280 - val_loss: 0.3603
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8863 - loss: 0.2731 - val_accuracy: 0.8743 - val_loss: 0.2909
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9046 - loss: 0.2354 - val_accuracy: 0.8891 - val_loss: 0.2717
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9078 - loss: 0.2261 - val_accuracy: 0.8570 - val_loss: 0.3282
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9138 - loss: 0.2131 - val_accuracy: 0.8931 - val_loss: 0.2627
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9236 - loss: 0.1938 - val_accuracy: 0.9000 - val_loss: 0.2585
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9280 - loss: 0.1817 - val_accuracy: 0.

In [57]:
ann_dim256_results, ann_dim256_probabilities, ann_dim256_predictions, ann_dim256_cm = evaluate_model(
    ann_dim256,
    X_test_integer,
    y_test,
    "ANN Dim 256",
    batch_size=64
)

ANN Dim 256 RESULTS
Accuracy : 0.89312
Precision: 0.91663
Recall   : 0.86581
F1 Score : 0.89050
ROC-AUC  : 0.95601

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8719    0.9206    0.8956      2470
    Positive     0.9166    0.8658    0.8905      2489

    accuracy                         0.8931      4959
   macro avg     0.8943    0.8932    0.8931      4959
weighted avg     0.8944    0.8931    0.8931      4959

Confusion Matrix
[[2274  196]
 [ 334 2155]]


In [58]:
ann_results_list.append(ann_dim256_results)

ann_df = pd.DataFrame(ann_results_list)
display(ann_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.750150,0.967120,0.519890,0.676250,0.952250
1,ANN EarlyStopping,0.894334,0.914033,0.871434,0.892225,0.955224
2,ANN Dropout,0.857633,0.945973,0.759743,0.842692,0.957139
3,ANN Batch Normalization,0.714257,0.914219,0.475291,0.625430,0.893157
4,ANN Learning Rate 0.0005,0.858036,0.936004,0.769787,0.844797,0.956050
5,ANN ReduceLR,0.892922,0.920894,0.860587,0.889720,0.959529
6,ANN Batch Size 32,0.895745,0.912898,0.875854,0.893992,0.953881
7,ANN Batch Size 128,0.897358,0.907072,0.886300,0.896566,0.957588
8,ANN SGD,0.496471,0.498650,0.593813,0.542087,0.502848
9,ANN RMSprop,0.831216,0.782297,0.919646,0.845429,0.918507


**ANN Sequence Length 300**

In [59]:
MAX_SEQUENCE_LENGTH_300 = 300

X_train_seq300 = tokenizer.texts_to_sequences(X_train)
X_val_seq300 = tokenizer.texts_to_sequences(X_val)
X_test_seq300 = tokenizer.texts_to_sequences(X_test)

X_train_300 = pad_sequences(
    X_train_seq300,
    maxlen=MAX_SEQUENCE_LENGTH_300,
    padding="post",
    truncating="post"
)

X_val_300 = pad_sequences(
    X_val_seq300,
    maxlen=MAX_SEQUENCE_LENGTH_300,
    padding="post",
    truncating="post"
)

X_test_300 = pad_sequences(
    X_test_seq300,
    maxlen=MAX_SEQUENCE_LENGTH_300,
    padding="post",
    truncating="post"
)

print("X_train:", X_train_300.shape)
print("X_val  :", X_val_300.shape)
print("X_test :", X_test_300.shape)

X_train: (39665, 300)
X_val  : (4958, 300)
X_test : (4959, 300)


In [60]:
ann_seq300 = Sequential([
    Input(
        shape=(300,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    GlobalAveragePooling1D(),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_seq300.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_seq300.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_10 (Embedding)        │ (None, 300, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_10     │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,864,833 (14.74 MB)

 Trainable params: 3,864,833 (14.74 MB)

 Non-trainable params: 0 (0.00 B)

In [61]:
history_ann_seq300 = ann_seq300.fit(
    X_train_300,
    y_train,
    validation_data=(X_val_300, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.7779 - loss: 0.4358 - val_accuracy: 0.8443 - val_loss: 0.3457
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8875 - loss: 0.2697 - val_accuracy: 0.8899 - val_loss: 0.2726
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9120 - loss: 0.2197 - val_accuracy: 0.8955 - val_loss: 0.2697
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9303 - loss: 0.1780 - val_accuracy: 0.8931 - val_loss: 0.2723
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9429 - loss: 0.1531 - val_accuracy: 0.8881 - val_loss: 0.3012
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9551 - loss: 0.1240 - val_accuracy: 0.8812 - val_loss: 0.3388
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9563 - loss: 0.1165 - val_accuracy: 0.8842 - val_loss: 0.3352
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9632 - loss: 0.1003 - val_accuracy: 0.

In [62]:
ann_seq300_results, ann_seq300_probabilities, ann_seq300_predictions, ann_seq300_cm = evaluate_model(
    ann_seq300,
    X_test_300,
    y_test,
    "ANN Sequence Length 300",
    batch_size=64
)

ANN Sequence Length 300 RESULTS
Accuracy : 0.86933
Precision: 0.90533
Recall   : 0.82603
F1 Score : 0.86387
ROC-AUC  : 0.93671

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8389    0.9130    0.8744      2470
    Positive     0.9053    0.8260    0.8639      2489

    accuracy                         0.8693      4959
   macro avg     0.8721    0.8695    0.8691      4959
weighted avg     0.8722    0.8693    0.8691      4959

Confusion Matrix
[[2255  215]
 [ 433 2056]]


In [63]:
ann_results_list.append(ann_seq300_results)

ann_df = pd.DataFrame(ann_results_list)
display(ann_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.750150,0.967120,0.519890,0.676250,0.952250
1,ANN EarlyStopping,0.894334,0.914033,0.871434,0.892225,0.955224
2,ANN Dropout,0.857633,0.945973,0.759743,0.842692,0.957139
3,ANN Batch Normalization,0.714257,0.914219,0.475291,0.625430,0.893157
4,ANN Learning Rate 0.0005,0.858036,0.936004,0.769787,0.844797,0.956050
5,ANN ReduceLR,0.892922,0.920894,0.860587,0.889720,0.959529
6,ANN Batch Size 32,0.895745,0.912898,0.875854,0.893992,0.953881
7,ANN Batch Size 128,0.897358,0.907072,0.886300,0.896566,0.957588
8,ANN SGD,0.496471,0.498650,0.593813,0.542087,0.502848
9,ANN RMSprop,0.831216,0.782297,0.919646,0.845429,0.918507


In [68]:
ann_df.to_csv("ann_model_comparison.csv", index=False)
print("saved")


saved


In [70]:
ann_batch128.save("best_ann_batch128.keras")
ann_batch128_history_df = pd.DataFrame(history_ann_batch128.history)
ann_batch128_history_df.to_csv("best_ann_batch128_training_history.csv", index=False)
print("saved")

saved
